# 卷积神经网络应用

欢迎来到课程4的第二个作业！在此 Notebook 中，你将：

- 使用 TF Keras Sequential API 创建情绪分类器
- 使用 TF Keras Functional API 构建 ConvNet 以识别手语数字

**在此作业之后，你将能够：**

- 在 TensorFlow 中构建并训练一个 ConvNet，解决 __binary__ 分类问题
- 在 TensorFlow 中构建并训练一个 ConvNet 以解决 __multiclass__ 分类问题
- 解释Sequential API 和Functional API 的不同用例

要完成这个作业，你应该已经熟悉TensorFlow。如果你不熟悉，请参阅课程 2 第三周的 **TensorFlow 教程**（“**改进深度神经网络**”）。

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/convolutional-neural-networks/supplement/DS4yP/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [1 - 依赖库](#1)
    - [1.1 - 加载数据并将数据拆分为训练/测试集](#1-1)
- [2 - TF Keras 中的层](#2)
- [3 - Sequential API](#3)
    - [3.1 - 创建 Sequential 模型](#3-1)
        - [练习 1 - happyModel](#ex-1)
    - [3.2 - 训练和评估模型](#3-2)
- [4 - Functional API](#4)
    - [4.1 - 加载 SIGNS 数据集](#4-1)
    - [4.2 - 将数据拆分为训练/测试集](#4-2)
    - [4.3 - 前向传播](#4-3)
        - [练习 2 - convolutional_model](#ex-2)
    - [4.4 - 训练模型](#4-4)
- [5 - History 对象](#5)
- [6 - 参考文献](#6)

<a name='1'></a>
## 1 - 依赖库

像往常一样，首先装入包裹。

In [ ]:
### v1.1

In [ ]:
import math
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import imread
import scipy
from PIL import Image
import pandas as pd
import tensorflow as tf
import tensorflow.keras.layers as tfl
from tensorflow.python.framework import ops
from cnn_utils import *
from test_utils import summary, comparator

%matplotlib inline
np.random.seed(1)

<a name='1-1'></a>
### 1.1 - 加载数据并将数据拆分为训练/测试集

你将在作业的这一部分中使用 Happy House 数据集，其中包含人脸图像。你的任务是建立一个卷积网络来确定图像中的人是否在微笑——因为他们只有在微笑的情况下才能进入房子！  

In [ ]:
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_happy_dataset()

# 归一化图像载体
X_train = X_train_orig/255.
X_test = X_test_orig/255.

# 重塑
Y_train = Y_train_orig.T
Y_test = Y_test_orig.T

print ("number of training examples = " + str(X_train.shape[0]))
print ("number of test examples = " + str(X_test.shape[0]))
print ("X_train shape: " + str(X_train.shape))
print ("Y_train shape: " + str(Y_train.shape))
print ("X_test shape: " + str(X_test.shape))
print ("Y_test shape: " + str(Y_test.shape))

你可以显示数据集中包含的图像。图像为 **64x64** 像素，采用 RGB 格式（3 通道）。

In [ ]:
index = 124
plt.imshow(X_train_orig[index]) #显示样本训练图像
plt.show()

<a name='2'></a>
## 2 - TF Keras 中的层 

在之前的作业中，你在NumPy中手动创建了层。在TF Keras中，你不必直接编写代码来创建层。相反，TF Keras 具有可供使用的预定义层。 

当你在 TF Keras 中创建层时，你正在创建一个函数，它接受一些输入并将其转换为稍后可以重复使用的输出。又好又简单！ 

<a name='3'></a>
## 3 - Sequential API

在之前的作业中，你使用 `numpy` 构建了辅助函数来了解卷积神经网络背后的机制。当今深度学习的大多数实际应用都是使用编程框架构建的，这些框架有许多内置的函数你可以简单地调用。 Keras 是构建在 TensorFlow 之上的高级抽象，它允许更加简化和优化的模型创建和训练。 

对于此作业的第一部分，你将使用 TF Keras 的Sequential API 创建一个模型，它允许你逐层构建，并且非常适合构建每层具有**恰好一个**输入张量和**一个**输出张量的模型。 

正如你将看到的，使用 Sequential API 既简单又直接，但仅适用于更简单、更直接的任务。稍后在本 Notebook 中，你将花一些时间使用更灵活、更强大的替代方案进行构建：Functional API。 
 

<a name='3-1'></a>
### 3.1 - 创建 Sequential 模型

如前所述，TensorFlow Keras Sequential API 可用于构建具有按顺序进行的层操作的简单模型。 

你还可以使用 `.add()` 方法向Sequential 模型增量添加层，或使用 `.pop()` 方法删除它们，就像在常规 Python 列表中一样。

实际上，你可以将Sequential 模型视为一个层列表。与 Python 列表一样，顺序层也是有序的，并且指定它们的顺序很重要。  如果你的模型是非线性的或包含具有多个输入或输出的层，则Sequential 模型不是正确的选择！

对于 Keras 中的任何层构造，你需要提前指定输入形状。这是因为在 Keras 中，权重的形状基于输入的形状。仅当模型第一次看到一些输入数据时才会创建权重。可以通过将层列表传递给 Sequential 构造函数来创建序列模型，就像你在下一个作业中所做的那样。

<a name='ex-1'></a>
### 练习 1 - happyModel

实现下面的 `happyModel` 函数来构建以下模型：`ZEROPAD2D -> CONV2D -> BATCHNORM -> RELU -> MAXPOOL -> FLATTEN -> DENSE`。寻求 [tf.keras.layers](https://www.tensorflow.org/api_docs/python/tf/keras/layers) 的帮助 

此外，为所有步骤插入以下参数：

 - [ZeroPadding2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/ZeroPadding2D)：填充3，输入形状64 x 64 x 3
 - [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D)：使用 32 个 7x7 卷积核，步长 1
 - [BatchNormalization](https://www.tensorflow.org/api_docs/python/tf/keras/layers/BatchNormalization)：用于轴 3
 - [ReLU](https://www.tensorflow.org/api_docs/python/tf/keras/layers/ReLU)
 - [MaxPool2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D)：使用默认参数
 - [Flatten](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten) 之前的输出。
 - 全连接（[Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)）层：应用具有 1 个神经元和 sigmoid 激活值的全连接层。 
 
 
 **提示：**
 
 使用 **tfl** 作为 **tensorflow.keras.layers** 的简写

In [ ]:
# GRADED FUNCTION: happyModel

def happyModel():
    """
    实现二元分类模型的前向传播：
    ZEROPAD2D -> CONV2D -> BATCHNORM -> RELU -> MAXPOOL -> FLATTEN -> DENSE
    
    请注意，为了简单起见和评分目的，你将对所有值进行硬编码
    例如步长和内核（卷积核）大小。 
    通常，函数应将这些值作为函数参数。
    
    参数：
    无

    返回：
    model -- TF Keras模型（包含整个训练过程信息的对象） 
    """
    model = tf.keras.Sequential([
            ## ZeroPadding2D，填充为 3，输入形状为 64 x 64 x 3
            
            ## Conv2D，具有 32 个 7x7 卷积核，步长为 1
            
            ## 轴 3 的 BatchNormalization
            
            ## ReLU
            
            ## 使用默认参数的最大池化 2D
            
            ## 展平层
            
            ## 具有 1 个输出单元的全连接层 & 'sigmoid' 激活值
            
            # YOUR CODE STARTS HERE
            
            
            # YOUR CODE ENDS HERE
        ])
    
    return model

In [ ]:
happy_model = happyModel()
# 打印每层的摘要
for layer in summary(happy_model):
    print(layer)
    
output = [['ZeroPadding2D', (None, 70, 70, 3), 0, ((3, 3), (3, 3))],
            ['Conv2D', (None, 64, 64, 32), 4736, 'valid', 'linear', 'GlorotUniform'],
            ['BatchNormalization', (None, 64, 64, 32), 128],
            ['ReLU', (None, 64, 64, 32), 0],
            ['MaxPooling2D', (None, 32, 32, 32), 0, (2, 2), (2, 2), 'valid'],
            ['Flatten', (None, 32768), 0],
            ['Dense', (None, 1), 32769, 'sigmoid']]
    
comparator(summary(happy_model), output)

#### 预期输出：

```
['ZeroPadding2D', (None, 70, 70, 3), 0, ((3, 3), (3, 3))]
['Conv2D', (None, 64, 64, 32), 4736, 'valid', 'linear', 'GlorotUniform']
['BatchNormalization', (None, 64, 64, 32), 128]
['ReLU', (None, 64, 64, 32), 0]
['MaxPooling2D', (None, 32, 32, 32), 0, (2, 2), (2, 2), 'valid']
['Flatten', (None, 32768), 0]
['Dense', (None, 1), 32769, 'sigmoid']
All tests passed!
```

模型创建后，可以选择优化器和损失函数并编译模型，为训练做好准备。当使用字符串 `accuracy` 指定评估指标时，Keras 会根据损失函数自动选择对应的准确率计算方式。有关编译设置的详细说明，请参阅[文档](https://www.tensorflow.org/api_docs/python/tf/keras/Model#compile)。

In [ ]:
happy_model.compile(optimizer='adam',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

是时候使用 `.summary()` 方法检查模型的参数了。这将显示你拥有的层类型、输出的形状以及每层中有多少个参数。 

In [ ]:
happy_model.summary()

<a name='3-2'></a>
### 3.2 - 训练和评估模型

创建模型、使用你选择的优化器和损失函数进行编译并对其内容进行健全性检查后，你现在就可以构建了！ 

调用 `.fit()` 即可训练模型。由于 TensorFlow 数据集已经完成分批，框架会处理 mini-batch、相关状态保存及反向传播计算。你也可以指定训练的 epoch 数；如果数据集尚未分批，还可以设置 mini-batch 大小。

In [ ]:
happy_model.fit(X_train, Y_train, epochs=10, batch_size=16)

完成后，只需使用 `.evaluate()` 与你的测试集进行评估即可。此函数将打印损失函数的值以及模型编译期间指定的性能指标。在本例中，分别为 `binary_crossentropy` 和 `accuracy`。

In [ ]:
happy_model.evaluate(X_test, Y_test)

简单吧？但是，如果你需要构建具有共享层、分支或多个输入和输出的模型怎么办？这就是 Sequential 无法帮助你的地方，虽然它非常简单但功能有限。 

接下来：进入函数式 API，它是你稍微复杂、高度灵活的朋友。  

<a name='4'></a>
## 4 - Functional API

欢迎来到作业的后半部分，你将使用 Keras 的灵活 [函数式 API](https://www.tensorflow.org/guide/keras/functional) 构建一个可以区分 6 个手语数字的 ConvNet。 

Functional API 支持非线性拓扑、共享层以及多个输入或输出。Sequential API 要求各层按顺序串联，而 Functional API 允许将模型构建为图，使各层之间的连接方式更加灵活。

下图对比了 Sequential 模型的顺序连接与跳跃连接；后者只是 Functional API 支持的多种连接方式之一。跳跃连接会跳过网络中的部分层，将输出直接传递给后续层。后续课程将进一步介绍这种结构。

<img src="images/seq_vs_func.png" style="width:350px;height:200px;">

<a name='4-1'></a>
### 4.1 - 加载 SIGNS 数据集

提醒一下，SIGNS 数据集是代表 0 到 5 数字的 6 个符号的集合。

In [ ]:
# 加载数据（标志）
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_signs_dataset()

<img src="images/SIGNS.png" style="width:800px;height:300px;">

下一个单元格将向你显示数据集中带标签图像的示例。请随意更改下面的 `index` 的值并重新运行以查看不同的示例。 

In [ ]:
# 数据集中的图像示例
index = 9
plt.imshow(X_train_orig[index])
print ("y = " + str(np.squeeze(Y_train_orig[:, index])))

<a name='4-2'></a>
### 4.2 - 将数据拆分为训练/测试集

在课程 2 中，你为此数据集构建了一个全连接网络。但由于这是一个图像数据集，因此对其应用 ConvNet 更为自然。

首先，让我们检查一下数据的形状。 

In [ ]:
X_train = X_train_orig/255.
X_test = X_test_orig/255.
Y_train = convert_to_one_hot(Y_train_orig, 6).T
Y_test = convert_to_one_hot(Y_test_orig, 6).T
print ("number of training examples = " + str(X_train.shape[0]))
print ("number of test examples = " + str(X_test.shape[0]))
print ("X_train shape: " + str(X_train.shape))
print ("Y_train shape: " + str(Y_train.shape))
print ("X_test shape: " + str(X_test.shape))
print ("Y_test shape: " + str(Y_test.shape))

<a name='4-3'></a>
### 4.3 - 前向传播

在TensorFlow中，内置了函数来为你实现卷积步骤。现在，你应该熟悉 TensorFlow 如何构建计算图。在 [函数式 API](https://www.tensorflow.org/guide/keras/functional) 中，你创建层图。这就是允许如此巨大的灵活性的原因。

但是，由于信息流位于单行上，因此也可以使用 Sequential API 定义以下模型。但不要偏离。我们希望你学习的是使用函数式 API。

通过创建一个输入节点函数作为可调用对象来开始构建层图：

- **input_img = tf.keras.Input(shape=input_shape):** 

然后，通过调用 `input_img` 对象上的层在层图中创建一个新节点： 

- **tf.keras.layers.Conv2D(filters= ... , kernel_size= ... , padding='same')(input_img):** 阅读有关 [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D) 的完整文档。

- **tf.keras.layers.MaxPool2D(pool_size=(f, f), strides=(s, s), padding='same'):** `MaxPool2D()` 使用大小为 (f, f) 的窗口和大小为 (s, s) 的步长对输入进行下采样，以在每个窗口上执行最大池化。  对于最大池化，你通常一次操作一个示例，一次操作一个通道。阅读有关 [MaxPool2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D) 的完整文档。

- **tf.keras.layers.ReLU():** 计算 Z 的元素 ReLU（可以是任何形状）。你可以阅读有关 [ReLU](https://www.tensorflow.org/api_docs/python/tf/keras/layers/ReLU) 的完整文档。

- **tf.keras.layers.Flatten()**：给定张量“P”，此函数获取批次中的每个训练（或测试）示例并将其展平为一维向量。  

    * 如果张量 P 的形状为 (batch_size,h,w,c)，则它返回形状为 (batch_size, k) 的展平张量，其中 $k=h \times w \times c$。  “k”等于除第一维度之外的所有维度大小的乘积。
    
    * 例如，给定一个尺寸为 [100, 2, 3, 4] 的张量，它将张量展平为形状 [100, 24]，其中 24 = 2 * 3 * 4。你可以阅读 [Flatten](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten) 上的完整文档。

- **tf.keras.layers.Dense(units= ... , activation='softmax')(F):** 给定扁平输入 F，它返回使用全连接层计算的输出。你可以阅读有关 [Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense) 的完整文档。

在上面的最后一个函数 (`tf.keras.layers.Dense()`) 中，全连接层自动初始化图中的权重，并在训练模型时继续训练它们。因此，在初始化参数时不需要初始化这些权重。

最后，在创建模型之前，你需要使用函数的最后一个组合（在本例中为 Dense 层）来定义输出： 

- **输出 = tf.keras.layers.Dense(units=6, activation='softmax')(F)**


#### 窗口、卷积核、池化窗口与池化

在本节中，kernel 与 filter 都指卷积核。Conv2D 的 filters 参数指定卷积核的数量，kernel_size 指定每个卷积核的空间尺寸。pooling（池化）则是在局部窗口内取最大值或平均值。

因此，`pool_size` 指定池化窗口的空间尺寸，作用类似于卷积层的 `kernel_size`；这里用 `(f,f)` 表示窗口大小。

池大小和内核大小在不同的对象中指的是同一件事 - 它们指的是发生操作的窗口的形状。 

<a name='ex-2'></a>
### 练习 2 - convolutional_model

实现下面的 `convolutional_model` 函数来构建以下模型：`CONV2D -> RELU -> MAXPOOL -> CONV2D -> RELU -> MAXPOOL -> FLATTEN -> DENSE`。使用上面的函数！ 

此外，为所有步骤插入以下参数：

 - [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D)：使用 8 个 4 x 4 卷积核，步长 1，填充为“SAME”
 - [ReLU](https://www.tensorflow.org/api_docs/python/tf/keras/layers/ReLU)
 - [MaxPool2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D)：使用 8 x 8 池化窗口尺寸和 8 x 8 步长，填充为'SAME'
 - **Conv2D**：使用 16 个 2 x 2 卷积核，步长 1，填充为'SAME'
 - **ReLU**
 - **MaxPool2D**：使用 4 x 4 池化窗口大小和 4 x 4 步长，填充为'SAME'
 - [Flatten](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten) 之前的输出。
 - 全连接（[Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)）层：应用具有 6 个神经元和 softmax 激活值的全连接层。 

In [ ]:
# GRADED FUNCTION: convolutional_model

def convolutional_model(input_shape):
    """
    为模型实现前向传播：
    CONV2D -> RELU -> MAXPOOL -> CONV2D -> RELU -> MAXPOOL -> FLATTEN -> DENSE
    
    请注意，为了简单起见和评分需要，你将对一些值进行硬编码
    例如步长和内核（卷积核）大小。 
    通常，函数应将这些值作为函数参数。
    
    参数：
    input_img -- 输入数据集，形状 (input_shape)

    返回：
    model -- TF Keras模型（包含整个训练过程信息的对象） 
    """

    input_img = tf.keras.Input(shape=input_shape)
    ## CONV2D：8 个卷积核 4x4，步长为 1，填充 'SAME'
    # Z1 = None
    ## ReLU
    # A1 = None
    ## MAXPOOL：窗口 8x8，步长 8，填充 'SAME'
    # P1 = None
    ## CONV2D：16 个卷积核 2x2，步长 1，填充 'SAME'
    # Z2 = None
    ## ReLU
    # A2 = None
    ## MAXPOOL：窗口 4x4，步长 4，填充 'SAME'
    # P2 = None
    ## 压扁
    # F = None
    ## 全连接层
    ## 输出层有 6 个神经元。提示：参数之一应该是 "activation='softmax'" 
    # outputs = None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    model = tf.keras.Model(inputs=input_img, outputs=outputs)
    return model

In [ ]:
conv_model = convolutional_model((64, 64, 3))
conv_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
conv_model.summary()
    
output = [['InputLayer', [(None, 64, 64, 3)], 0],
        ['Conv2D', (None, 64, 64, 8), 392, 'same', 'linear', 'GlorotUniform'],
        ['ReLU', (None, 64, 64, 8), 0],
        ['MaxPooling2D', (None, 8, 8, 8), 0, (8, 8), (8, 8), 'same'],
        ['Conv2D', (None, 8, 8, 16), 528, 'same', 'linear', 'GlorotUniform'],
        ['ReLU', (None, 8, 8, 16), 0],
        ['MaxPooling2D', (None, 2, 2, 16), 0, (4, 4), (4, 4), 'same'],
        ['Flatten', (None, 64), 0],
        ['Dense', (None, 6), 390, 'softmax']]
    
comparator(summary(conv_model), output)

Sequential API 和Functional API 均返回 TF Keras 模型对象。唯一的区别是对象模型内部如何处理输入！ 

<a name='4-4'></a>
### 4.4 - 训练模型

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train)).batch(64)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, Y_test)).batch(64)
history = conv_model.fit(train_dataset, epochs=100, validation_data=test_dataset)

<a name='5'></a>
## 5 - History 对象 

历史对象是 `.fit()` 操作的输出，并提供内存中所有损失和指标值的记录。它存储为字典，你可以在 `history.history` 检索： 

In [ ]:
history.history

现在使用 `history.history` 可视化随时间推移的损失： 

In [ ]:
# history.history["loss"] 条目是一个字典，其中包含与
# 模型经过训练。 
df_loss_acc = pd.DataFrame(history.history)
df_loss= df_loss_acc[['loss','val_loss']]
df_loss.rename(columns={'loss':'train','val_loss':'validation'},inplace=True)
df_acc= df_loss_acc[['accuracy','val_accuracy']]
df_acc.rename(columns={'accuracy':'train','val_accuracy':'validation'},inplace=True)
df_loss.plot(title='Model loss',figsize=(12,8)).set(xlabel='Epoch',ylabel='Loss')
df_acc.plot(title='Model Accuracy',figsize=(12,8)).set(xlabel='Epoch',ylabel='Accuracy')

**恭喜**！你已经完成了作业并构建了两个模型：一个可以识别微笑，另一个可以在测试集上识别 SIGN 语言，准确率接近 80%。除此之外，你现在还了解了两个 Keras API 的应用：Sequential API 和Functional API。干得好！ 

到目前为止，你已经了解了一些 Functional API 的工作原理，并且可能已经看到了其中的可能性。在你的下一个作业中，当你有机会使用 ResNet 构建非常深的 ConvNet 时，你将真正感受到它的强大功能！ 

<a name='6'></a>
## 6 - 参考文献

我们始终鼓励你阅读官方文档。为此，你可以在此处找到Sequential API 和Functional API 的文档： 

https://www.tensorflow.org/guide/keras/sequential_model

https://www.tensorflow.org/guide/keras/functional